# 01b · Translation quality check, human review, final files

Run after notebook 01. Still no model is run on the evaluation set.

- **Part A** back-translates every Hausa segment with a *different* NLLB model (`QE_MODEL`), scores the round trip (chrF1), flags suspect segments, and exports a review sheet with evaluation items only.
- **You** review the flagged segments (keep / english / edit).
- **Part B** applies your decisions, auto-applies English fallback to flagged training-pool segments, rebuilds the translate-test condition, exports the two rater sheets, and extends the manifest.

Stop after Part A and fill in the sheet before running Part B.

## 0 · Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
REPO_DIR = "/content/hausa-med-qa"
import os
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/avvas200/hausa-med-qa.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
!pip install -q -r requirements.txt

In [ ]:
import json, random, time
from collections import Counter
import pandas as pd
import torch
from src import config as C, data as D, qe as Q
from src.translate import Translator, translate_segments, _load

T = C.TRANSLATION_TAG
eval_en = D.read_jsonl(C.DATA / "eval_en.jsonl")
pool_en = D.read_jsonl(C.DATA / "train_pool_en.jsonl")
eval_by_id = {r["id"]: r for r in eval_en}
eval_segs, pool_segs = D.segments(eval_en), D.segments(pool_en)

# Reload translations and statuses from notebook 01's cache (no model needed:
# everything is cached, so tr=None is never called).
eval_ha_map, eval_ha_status = translate_segments(None, eval_segs, C.CACHE / f"eval_en2ha_{T}.jsonl", C.EN, C.HA, progress=False)
pool_ha_map, pool_ha_status = translate_segments(None, pool_segs, C.CACHE / f"pool_en2ha_{T}.jsonl", C.EN, C.HA, progress=False)
print(Counter(eval_ha_status.values()), Counter(pool_ha_status.values()))

# Part A · Quality check

## A1 · Independent back-translation with the QE model

In [ ]:
to_check = lambda status: [k for k, s in status.items() if s in ("translated", "retried")]
eval_keys, pool_keys = to_check(eval_ha_status), to_check(pool_ha_status)

# Load the QE model only if something is not cached yet (fast re-runs after a restart)
pending = [k for k in eval_keys if (k, eval_ha_map[k]) not in _load(C.CACHE / f"eval_ha2en_{C.QE_TAG}.jsonl")] +           [k for k in pool_keys if (k, pool_ha_map[k]) not in _load(C.CACHE / f"pool_ha2en_{C.QE_TAG}.jsonl")]
qe_tr = Translator(C.QE_MODEL) if pending else None
print("segments to back-translate:", len(pending))
eval_back, _ = translate_segments(qe_tr, {k: eval_ha_map[k] for k in eval_keys},
                                  C.CACHE / f"eval_ha2en_{C.QE_TAG}.jsonl", C.HA, C.EN)
pool_back, _ = translate_segments(qe_tr, {k: pool_ha_map[k] for k in pool_keys},
                                  C.CACHE / f"pool_ha2en_{C.QE_TAG}.jsonl", C.HA, C.EN)
del qe_tr; torch.cuda.empty_cache()

## A2 · Score and flag

In [ ]:
def score(keys, segs, ha, back):
    rows = []
    for k in keys:
        field = "question" if k.endswith("|q") else "option"
        s, reasons = Q.flag_reasons(segs[k], ha[k], back[k], field)
        rows.append({"key": k, "field": field, "en": segs[k], "ha": ha[k],
                     "back_qe": back[k], "chrf1": s, "reasons": reasons, "flagged": bool(reasons)})
    return pd.DataFrame(rows)

eval_qe = score(eval_keys, eval_segs, eval_ha_map, eval_back)
pool_qe = score(pool_keys, pool_segs, pool_ha_map, pool_back)

for name, df in (("EVAL", eval_qe), ("POOL", pool_qe)):
    print(f"--- {name} ---")
    print(df.groupby("field").agg(n=("key", "size"), flagged=("flagged", "sum"),
                                  median_chrf=("chrf1", "median")).to_string())
    print("reasons:", Counter(r for rs in df.reasons for r in rs))
print("\nEvaluation items with >=1 flagged segment:",
      eval_qe[eval_qe.flagged].key.str.split("|").str[0].nunique(), "of", len(eval_en))

In [ ]:
# chrF1 distribution for options: helps judge whether the threshold gives a
# manageable review load. Adjust C.QE_MIN_CHRF_* here only, before exporting.
bins = [0, 20, 40, 60, 80, 101]
print(pd.cut(eval_qe[eval_qe.field == "option"].chrf1, bins, right=False).value_counts().sort_index())
for lo, hi in [(0, 20), (40, 60), (60, 80)]:
    sub = eval_qe[(eval_qe.field == "option") & (eval_qe.chrf1 >= lo) & (eval_qe.chrf1 < hi)]
    print(f"\n== option chrF1 in [{lo},{hi}) sample ==")
    for r in sub.sample(min(5, len(sub)), random_state=0).itertuples():
        print(f"  EN: {r.en}\n  HA: {r.ha}\n  QE back: {r.back_qe}\n")

## A3 · Export review sheet (evaluation items only)

In [ ]:
sheet = Q.adjudication_sheet(eval_qe[eval_qe.flagged], eval_by_id)
sheet_path = C.DATA / "adjudication_eval.csv"
sheet.to_csv(sheet_path, index=False, encoding="utf-8-sig")
print(len(sheet), "segments to review ->", sheet_path)
print(sheet.field.value_counts().to_string())

### How to fill the sheet

Open `adjudication_eval.csv` in Google Sheets. For each row, set `decision`:

| decision | when | allowed for |
|---|---|---|
| `keep` | Hausa is correct, even if the back-translation is a paraphrase (e.g. *hawan jini* for hypertension) | options, questions |
| `english` | Hausa is wrong, and a Hausa speaker would naturally say the English term anyway | options |
| `edit` | Hausa is wrong and you can write a correct Hausa version; put it in `edited_ha` | options, questions |

A blank `decision` takes the `suggested` value (options: english, questions: keep) and is logged as a default. **Don't look at any model outputs while doing this, and don't change which option is correct.** `is_gold_option` is shown only so that you take extra care with those rows; an error on the gold option matters most.

When you're done, download it as CSV and save it to Drive as `hausa-med-qa/data/adjudication_eval_done.csv`.

# Part B · Apply decisions and build final files
If the runtime restarted while you were reviewing, first re-run everything from the top through A2. It's fast, because all back-translations are cached. Skip A3 so the export isn't rewritten; it would be identical anyway.

In [ ]:
dec = Q.read_decisions(C.DATA / "adjudication_eval_done.csv")
print(dec.decision.value_counts().to_string(), "\ndefaulted:", int(dec.defaulted.sum()))
exported = pd.read_csv(C.DATA / "adjudication_eval.csv", dtype=str, encoding="utf-8-sig")
assert set(dec.key) == set(exported.key), "Completed sheet rows don't match the exported sheet"

In [ ]:
# Final Hausa evaluation segments + per-segment status for sensitivity analyses
final_ha, final_status = dict(eval_ha_map), dict(eval_ha_status)
for r in dec.itertuples(index=False):
    if r.decision == "english":
        final_ha[r.key], final_status[r.key] = eval_segs[r.key], "fallback_en_reviewed"
    elif r.decision == "edit":
        final_ha[r.key], final_status[r.key] = r.edited_ha.strip(), "post_edited"
    else:
        final_status[r.key] = "kept_after_review"

# Training pool: no human review; flagged segments fall back to English automatically
pool_final, pool_final_status = dict(pool_ha_map), dict(pool_ha_status)
for k in pool_qe[pool_qe.flagged].key:
    pool_final[k], pool_final_status[k] = pool_segs[k], "fallback_en_auto"

print("eval :", Counter(final_status.values()))
print("pool :", Counter(pool_final_status.values()))

In [ ]:
# Translate-test (H2) with the main 3.3B model. Segments kept in English are
# copied; unchanged Hausa segments reuse notebook 01's cache; only post-edited
# segments are newly translated.
english_like = {"passthrough", "fallback_en", "fallback_en_reviewed"}
copy_keys = [k for k, s in final_status.items() if s in english_like]
back_in = {k: (eval_segs[k] if k in copy_keys else v) for k, v in final_ha.items()}
cache_ha2en = C.CACHE / f"eval_ha2en_{T}.jsonl"
cached = _load(cache_ha2en)
need = [k for k, v in back_in.items() if (k, v) not in cached and k not in copy_keys]
print("segments needing new back-translation:", len(need))
tr = Translator(C.NLLB_MODEL) if need else None
ha2en_map, ha2en_status = translate_segments(tr, back_in, cache_ha2en, C.HA, C.EN, copy_keys=copy_keys)

In [ ]:
eval_ha = D.apply_translations(eval_en, final_ha, "ha")
eval_ha2en = D.apply_translations(eval_en, ha2en_map, "ha2en")
pool_ha = D.apply_translations(pool_en, pool_final, "ha")
D.write_jsonl(eval_ha, C.DATA / "eval_ha.jsonl")
D.write_jsonl(eval_ha2en, C.DATA / "eval_ha2en.jsonl")
D.write_jsonl(pool_ha, C.DATA / "train_pool_ha.jsonl")

json.dump({"eval": final_status, "pool": pool_final_status},
          open(C.DATA / "segment_status.json", "w"))
fallback_items = sorted({k.split("|")[0] for k, s in final_status.items()
                         if s in ("fallback_en", "fallback_en_reviewed")})
json.dump(fallback_items, open(C.DATA / "fallback_item_ids.json", "w"))
print("evaluation items with >=1 English-fallback option:", len(fallback_items), "of", len(eval_en))

## B2 · Rater sheets (random 75 items, from the final Hausa)
Rater 1 is you; rater 2 should be someone who did **not** do the review above. Scale for `adequacy`: 1 = meaning lost, 2 = major errors, 3 = partly preserved, 4 = minor errors, 5 = fully preserved. `term_error` = 1 if a medical term was mistranslated; an English term kept where Hausa speakers would use it is **not** an error.

In [ ]:
rng = random.Random(C.SEED + 2)
val_ids = rng.sample([r["id"] for r in eval_en], C.N_HUMAN_VALIDATION)
by_ha = {r["id"]: r for r in eval_ha}
rows = []
for i in val_ids:
    en, ha = eval_by_id[i], by_ha[i]
    row = {"id": i, "subject": en["subject"], "question_en": en["question"], "question_ha": ha["question"]}
    for j, L in enumerate(C.LETTERS):
        row[f"option_{L}_en"], row[f"option_{L}_ha"] = en["options"][j], ha["options"][j]
    row.update({"adequacy_1to5": "", "term_error_0or1": "", "notes": ""})
    rows.append(row)
rs = pd.DataFrame(rows)
for rater in ("rater1", "rater2"):
    rs.to_csv(C.DATA / f"validation_{rater}.csv", index=False, encoding="utf-8-sig")
json.dump(val_ids, open(C.DATA / "validation_ids.json", "w"))
reviewed_in_sample = sum(any(k.startswith(i + "|") for k in dec.key) for i in val_ids)
print("rater sheets written; items in the sample that you also reviewed:", reviewed_in_sample)

## B3 · Extend the manifest

In [ ]:
m = json.load(open("results/data_manifest.json"))
m["qe"] = {
    "model": C.QE_MODEL,
    "thresholds": {"option_chrf1": C.QE_MIN_CHRF_OPTION, "question_chrf1": C.QE_MIN_CHRF_QUESTION},
    "eval_flagged": {f: int(v) for f, v in eval_qe.groupby("field").flagged.sum().items()},
    "pool_flagged": {f: int(v) for f, v in pool_qe.groupby("field").flagged.sum().items()},
    "eval_decisions": dec.decision.value_counts().to_dict(),
    "eval_decisions_defaulted": int(dec.defaulted.sum()),
}
m["final_status"] = {"eval": dict(Counter(final_status.values())),
                     "pool": dict(Counter(pool_final_status.values())),
                     "eval_ha2en": dict(Counter(ha2en_status.values()))}
m["items_with_fallback"] = len(fallback_items)
m["updated"] = time.strftime("%Y-%m-%d %H:%M:%S")
files = ["eval_en.jsonl", "eval_ha.jsonl", "eval_ha2en.jsonl", "train_pool_en.jsonl",
         "train_pool_ha.jsonl", "validation_ids.json", "fallback_item_ids.json",
         "segment_status.json", "adjudication_eval_done.csv"]
m["sha256"] = {f: D.sha256(C.DATA / f) for f in files}
json.dump(m, open("results/data_manifest.json", "w"), indent=2)
json.dump(m, open(C.RESULTS / "data_manifest.json", "w"), indent=2)
print(json.dumps({k: m[k] for k in ("qe", "final_status", "items_with_fallback")}, indent=2))

## Next
1. Send out the rater sheets.
2. Commit `results/data_manifest.json`, and copy `adjudication_eval_done.csv` into `results/` so the review is auditable.
3. When ratings are back: fill the remaining prereg blanks, commit, and record the hash. **That commit is the freeze.**